# Индекс Биг Мака: расчёты

Итоговый проект, курс «Технологии анализа и визуализации экономической информации».

Данные:
- The Economist, Big Mac index: github.com/TheEconomist/big-mac-data
- «Вкусно – и точка» (цена «Биг Хита»): vkusnoitochka.ru/menu/burgery-i-rolly
- Курсы валют: ЦБ РФ (cbr.ru)
- ВВП на душу населения: Всемирный банк (2025)


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.offline as po

# 1. Загружаем данные The Economist
print("1. ЗАГРУЗКА ДАННЫХ")
df = pd.read_csv('data/big-mac-full-index.csv', parse_dates=['date'])
print(f"Размер: {df.shape}")
print(f"Диапазон дат: {df.date.min().date()} - {df.date.max().date()}")
print(f"Стран в наборе: {df.iso_a3.nunique()}")
df.head()


In [ ]:
# 2. Цены Биг Мака в США и России
print("\n2. ЦЕНЫ В США И РОССИИ")
usa = df[df.iso_a3 == 'USA'].sort_values('date')
rus = df[df.iso_a3 == 'RUS'].sort_values('date')

print("США:")
display(usa[['date', 'local_price', 'dollar_ex', 'dollar_price']].tail(4))

print("Россия (данные The Economist до 2022 года):")
display(rus[['date', 'local_price', 'dollar_ex', 'dollar_price', 'USD_raw']])


## Пересчёт по цене «Биг Хита»

- 12.12.2022 – 165 ₽ (запуск)
- до конца 2025 – 165 ₽
- с 2026 – 201 ₽ (минимум по стране)
- сентябрь 2026, Москва – 255 ₽


In [ ]:
# 3. Собираю цену «Биг Хита» и курсы ЦБ на даты наблюдений
print("\n3. ПЕРЕСЧЁТ ПО ЦЕНЕ БИГ ХИТА")
bighit = pd.DataFrame([
    ('2022-12-13', 165, 62.7674),   # запуск 12.12.2022
    ('2023-08-29', 165, 95.4717),
    ('2024-08-07', 165, 85.1646),
    ('2025-07-10', 165, 78.1727),
    ('2025-10-17', 165, 79.0839),
    ('2026-02-11', 201, 77.2091),
    ('2026-09-29', 255, 84.4075),   # Москва, сентябрь 2026
], columns=['date', 'local_price', 'dollar_ex'])
bighit['date'] = pd.to_datetime(bighit['date'])

# цена Биг Мака в США из набора The Economist по ближайшей дате
usa_ref = usa[['date', 'dollar_price']].rename(columns={'dollar_price': 'usa_dollar'})
bighit = pd.merge_asof(bighit, usa_ref, on='date', direction='nearest')

bighit['dollar_price'] = bighit.local_price / bighit.dollar_ex
bighit['raw_index'] = bighit.dollar_price / bighit.usa_dollar - 1

bighit[['date', 'local_price', 'dollar_ex', 'dollar_price', 'usa_dollar', 'raw_index']].style.format({
    'local_price': '{:.0f} ₽',
    'dollar_ex': '{:.2f}',
    'dollar_price': '{:.2f} $',
    'usa_dollar': '{:.2f} $',
    'raw_index': '{:+.1%}',
})


## Итог на 29.09.2026

Индекс = (цена бургера в долларах в стране ÷ цена в США) − 1.


In [ ]:
# 4. Индекс для России на последнюю дату
print("\n4. ИТОГ НА 29.09.2026")
last = bighit.iloc[-1]
print(f"Цена «Биг Хита», Москва:      {last.local_price:.0f} ₽")
print(f"Курс ЦБ РФ:                   {last.dollar_ex:.2f} ₽/$")
print(f"Цена в долларах:              {last.dollar_price:.2f} $")
print(f"Биг Мак в США (июль 2026):    {last.usa_dollar:.2f} $")
print(f"Индекс для России:            {last.raw_index:+.1%}")

world = df[(df.date == df.date.max()) & (df.iso_a3 != 'EUZ')].copy()
n_cheaper = int((world.USD_raw < last.raw_index).sum())
print(f"Стран с более дешёвым бургером: {n_cheaper} из {len(world)}")


## Сборка дашборда


In [ ]:
# 5. Собираю все графики дашборда
print("\n5. СБОРКА ДАШБОРДА")
%run build.py

map_fig.show()

print("=" * 50)
print("РАСЧЁТЫ И СБОРКА ДАШБОРДА ЗАВЕРШЕНЫ!")
print("=" * 50)


## Вывод

Рубль недооценён примерно на 51%: «Биг Хит» в Москве стоит 255 ₽, или 3,02 $, против
6,22 $ за Биг Мак в США.

Россия – 7-я с конца из 53 стран по дешевизне бургера и 14-я из 29 по числу бургеров
на среднюю чистую зарплату (391 бургер в месяц).
